[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wdittaya/AI-for-digital-health-course/blob/main/lab07_document_processing.ipynb)

# Lab 7: การประมวลผลเอกสารและการสกัดสารสนเทศทางคลินิก (Document Processing & Clinical Information Extraction)

**รายวิชา:** 3099704 AI for Digital Health (2026) · **คาบเรียน:** วันเสาร์ที่ 17 ตุลาคม 2026 (ช่วงเช้า) · **เวลาโดยประมาณ:** 60–90 นาที

## วัตถุประสงค์การเรียนรู้

- สกัดข้อความจากเอกสารทางคลินิกที่เป็นภาพสแกนหรือ PDF ด้วยการรู้จำอักขระด้วยแสง (OCR) โดยใช้ Tesseract
- ทำความสะอาดและจัดโครงสร้างข้อความจากเอกสารที่ไม่เป็นระเบียบ
- สกัดสารสนเทศทางคลินิก (ผลการตรวจทางห้องปฏิบัติการ และรายการยา) ด้วยกฎ (rules) และอธิบายประโยชน์ของโมเดลประมวลผลภาษาธรรมชาติ (NLP) ที่เรียนรู้จากข้อมูล
- สร้างระเบียนข้อมูลแบบมีโครงสร้าง (structured record) จากเอกสารที่ไม่มีโครงสร้าง และประเมินความถูกต้องของผลลัพธ์

> **แนวทางการทำปฏิบัติการ** (งานเดี่ยว)
>
> - ✅ **ส่วนหลัก — สำหรับผู้เรียนทุกคน ไม่จำเป็นต้องมีพื้นฐานการเขียนโปรแกรม:** ให้รันเซลล์ (cell) ตามลำดับจากบนลงล่าง โดยคลิกที่เซลล์แล้วกด **Shift+Enter** หรือคลิกปุ่ม ▶ แบบฝึกหัด ⭐ **Exercise** แต่ละข้อกำหนดให้แก้ไขเพียงเล็กน้อย เช่น เพิ่มชื่อในรายการ หรือพิมพ์ข้อความหนึ่งบรรทัด ถัดจากแบบฝึกหัดทุกข้อมีเซลล์ ✅ **เฉลย** ผู้เรียนควรทำแบบฝึกหัดด้วยตนเองก่อน หากไม่สามารถดำเนินการต่อได้ ให้รันเซลล์เฉลยแล้วทำขั้นตอนถัดไป จากนั้นตอบคำถาม ✍️ ในเซลล์ข้อความ **คำตอบดังกล่าวเป็นชิ้นงานที่ต้องส่ง**
> - 🏆 **โจทย์ท้าทายบนตารางอันดับ — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้:** อยู่ท้ายโน้ตบุ๊ก (notebook) เป็นการทำ OCR กับรายงานผลการตรวจทางห้องปฏิบัติการสังเคราะห์ 20 ฉบับ และรับคะแนนโดยอัตโนมัติ ไม่นับเป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง
>
> ปฏิบัติการนี้ทำงานได้บน [Google Colab](https://colab.research.google.com) รุ่นไม่มีค่าใช้จ่าย ด้วย runtime แบบ CPU ตามปกติ **โดยไม่จำเป็นต้องใช้ GPU** หากพบปัญหา สามารถสอบถามผู้สอนผ่านช่องแชตของ Zoom


In [ ]:
# @title ⚙️ การตั้งค่า — โปรดรันเซลล์นี้ก่อน (NAME และ JOIN_CODE ใช้เฉพาะโจทย์ท้าทายบนตารางอันดับ ซึ่งไม่บังคับ)
!pip -q install gradio_client huggingface_hub

HF_OWNER  = "wdittaya"     # บัญชี Hugging Face ที่ใช้โฮสต์ตารางอันดับของรายวิชา (ผู้สอนเป็นผู้กำหนด)
LAB_ID    = "lab07"
NAME      = ""            # ไม่บังคับ: ชื่อที่จะแสดงบนตารางอันดับแบบสาธารณะ (สามารถใช้นามแฝงได้)
JOIN_CODE = ""            # ไม่บังคับ: รหัสเข้าร่วมซึ่งผู้สอนประกาศในการเรียนการสอนผ่าน Zoom

SPACE_ID  = f"{HF_OWNER}/aidh-leaderboard"
DATA_REPO = f"{HF_OWNER}/aidh-lab-data"

from huggingface_hub import hf_hub_download

def lab_data(path):
    """ดาวน์โหลดไฟล์ข้อมูลทดสอบสาธารณะของรายวิชา 1 ไฟล์ (เช่น 'pima/test_ids.csv') และคืนค่าตำแหน่งไฟล์ในเครื่อง"""
    return hf_hub_download(DATA_REPO, path, repo_type="dataset")

def submit_to_leaderboard(path, lab=LAB_ID):
    """ส่งไฟล์ผลการทำนายไปยังตารางอันดับของรายวิชา และแสดงคะแนนพร้อมอันดับ"""
    from gradio_client import Client, handle_file
    assert NAME and JOIN_CODE, "โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่าก่อน (จำเป็นเฉพาะการส่งผลไปยังตารางอันดับ)"
    client = Client(SPACE_ID, verbose=False)
    print(client.predict(lab=lab, name=NAME, code=JOIN_CODE, file=handle_file(path), api_name="/submit"))

print(f"Leaderboard: https://huggingface.co/spaces/{SPACE_ID}  |  lab={LAB_ID}  name={NAME or '(ยังไม่ได้ระบุ — ตารางอันดับไม่บังคับ)'}")

## ความสำคัญของการประมวลผลเอกสาร

ข้อมูลสุขภาพจำนวนมากอยู่ในรูปแบบ PDF ภาพสแกน และข้อความอิสระ (free text) เช่น ใบส่งตัว รายงานผลการตรวจทางห้องปฏิบัติการ และสรุปการจำหน่ายผู้ป่วย (discharge summary) การแปลงข้อมูลเหล่านี้เป็นข้อมูลแบบมีโครงสร้าง ซึ่งคอมพิวเตอร์สามารถเรียงลำดับ ระบุค่าผิดปกติ และนำเสนอเป็นกราฟได้ จึงเป็นงานพื้นฐานที่สำคัญของสุขภาพดิจิทัล

กระบวนการ (pipeline) ในปฏิบัติการนี้ประกอบด้วย 3 ขั้นตอน ดังนี้

1. **OCR** (*Optical Character Recognition* — การรู้จำอักขระด้วยแสง) คือการใช้โปรแกรมอ่านภาพของหน้าเอกสารและแปลงเป็นข้อความ ปฏิบัติการนี้ใช้ **Tesseract** ซึ่งเป็นโปรแกรม OCR แบบโอเพนซอร์ส
2. **การสกัดสารสนเทศ (information extraction)** คือการใช้กฎค้นหาส่วนที่ต้องการในข้อความ ได้แก่ ชื่อการตรวจ ผลการตรวจ หน่วย ชื่อยา และขนาดยา
3. **การตรวจสอบความถูกต้อง** คือการเปรียบเทียบข้อมูลที่สกัดได้กับคำตอบที่ถูกต้อง เพื่อวัดความแม่นยำและตรวจหาการอ่านค่าผิดที่อาจเป็นอันตรายต่อผู้ป่วย

ให้รันเซลล์ถัดไปเพื่อติดตั้ง Tesseract (ใช้เวลาประมาณ 1 นาที)


In [ ]:
# ติดตั้งโปรแกรม OCR Tesseract และไลบรารี Python สำหรับเรียกใช้งาน (ประมาณ 1 นาที; ใช้ runtime แบบ CPU ได้)
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install pytesseract pillow
import re, difflib
import pytesseract, numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image, ImageDraw, ImageFont
print("เวอร์ชันของ Tesseract:", pytesseract.get_tesseract_version())


## ขั้นตอนที่ 1 — รายงานผลการตรวจทางห้องปฏิบัติการที่ผ่านการสแกน (ข้อมูลสังเคราะห์)

เซลล์ถัดไปสร้างภาพรายงานผลการตรวจทางห้องปฏิบัติการ เพื่อให้ผู้เรียนทุกคนใช้ข้อมูลนำเข้าชุดเดียวกัน ผู้ป่วยในรายงานเป็นบุคคลสมมติ **ห้ามใช้เอกสารของผู้ป่วยจริง (ข้อมูลสุขภาพส่วนบุคคล หรือ PHI) ในโน้ตบุ๊กสาธารณะ** ข้อความในรายงานเป็นภาษาอังกฤษตามรูปแบบของรายงานผลการตรวจจริง


In [ ]:
# สร้างภาพ 'รายงานผลการตรวจทางห้องปฏิบัติการ' สังเคราะห์ (ไม่มีข้อมูลผู้ป่วยจริง)
report_text = """CENTRAL HOSPITAL - LABORATORY REPORT
Patient: John Doe    MRN: 001234    Age: 57    Sex: M
Date: 2026-10-17

TEST                RESULT     UNIT       REFERENCE
Hemoglobin          11.2       g/dL       13.5-17.5
Glucose (fasting)   142        mg/dL      70-99
Creatinine          1.8        mg/dL      0.7-1.3
LDL cholesterol     165        mg/dL      <100

Medications: Metformin 500 mg BID; Atorvastatin 20 mg OD
Impression: Anemia, hyperglycemia, elevated LDL."""

try:
    font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf", 20)   # มีอยู่ใน Colab
except OSError:
    font = ImageFont.load_default()
img = Image.new("RGB", (900, 520), "white")
d = ImageDraw.Draw(img)
d.multiline_text((20, 20), report_text, fill="black", spacing=8, font=font)
img.save("report.png")
plt.figure(figsize=(9, 6)); plt.imshow(img); plt.axis("off"); plt.show()


## ขั้นตอนที่ 2 — การแปลงภาพเป็นข้อความด้วย OCR

สำหรับคอมพิวเตอร์ ไฟล์ `report.png` เป็นเพียงตารางของจุดสี (pixel) ระบบจึงไม่สามารถค้นหาคำว่า "Creatinine" ในภาพได้โดยตรง OCR ทำหน้าที่รู้จำตัวอักษรและแปลงภาพเป็นข้อความธรรมดา


In [ ]:
# ⭐ Exercise 1 — แบบฝึกหัดที่ 1: การทำ OCR กับภาพ
# ให้แทนที่ ... ด้วยโค้ดบรรทัดต่อไปนี้ (สามารถคัดลอกและวางได้):
#     pytesseract.image_to_string(Image.open("report.png"), config="--psm 6")
# คำแนะนำ: Image.open(...) ใช้เปิดไฟล์ภาพ ส่วน image_to_string(...) ใช้อ่านข้อความจากภาพ
#          "--psm 6" กำหนดให้ Tesseract ถือว่าหน้าเอกสารเป็นข้อความหลายบรรทัดในกลุ่มเดียว (เหมาะกับตาราง)
ocr_text = ...    # TODO
print(ocr_text)


In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
ocr_text = pytesseract.image_to_string(Image.open("report.png"), config="--psm 6")
print(ocr_text)


In [ ]:
# เปรียบเทียบข้อความจาก OCR กับข้อความต้นฉบับในภาพ (ไม่นับช่องว่างส่วนเกิน)
def squash(s):
    return [" ".join(line.split()) for line in s.splitlines() if line.strip()]

original, ocr_lines = squash(report_text), squash(ocr_text)
print(f"ต้นฉบับ {len(original)} บรรทัด, OCR อ่านได้ {len(ocr_lines)} บรรทัด\n")
n_diff = 0
for line in difflib.unified_diff(original, ocr_lines, "typed", "OCR", lineterm="", n=0):
    if line.startswith(("-", "+")) and not line.startswith(("---", "+++")):
        print(line); n_diff += 1
print("✅ ข้อความจาก OCR ตรงกับต้นฉบับทุกตัวอักษร" if n_diff == 0 else "\n(- = ข้อความในเอกสาร, + = ข้อความที่ OCR อ่านได้)")


💡 **คำอธิบายผลลัพธ์**

Tesseract แปลงภาพเป็นข้อความ โดยคอลัมน์ของตารางถูกแทนด้วยช่องว่างเพียงหนึ่งช่อง กล่าวคือ OCR ให้ผลลัพธ์เป็น *ข้อความ* มิใช่ตาราง สำหรับภาพที่สร้างด้วยคอมพิวเตอร์และมีความคมชัดเช่นนี้ OCR มักอ่านได้ถูกต้องทั้งหมด แต่ภาพสแกนจริง (ภาพเอียง แฟกซ์ มีตราประทับ หรือลายมือ) มักเกิดข้อผิดพลาด เช่น `O`↔`0`, `l`↔`1`, `B`↔`8` ผลกระทบของข้อผิดพลาดดังกล่าวจะแสดงในขั้นตอนที่ 6


✍️ **คำถามที่ 1** จากผลการเปรียบเทียบข้างต้น OCR อ่านรายงานฉบับนี้ผิดพลาดหรือไม่ อย่างไร และโปรดระบุ **ลักษณะของเอกสารทางคลินิกในหน่วยงานของผู้เรียนอย่างน้อย 3 ประการ** ที่ทำให้การทำ OCR ยากขึ้น (เช่น ลายมือ ตราประทับ แฟกซ์ ตาราง หรือข้อความภาษาไทยปนภาษาอังกฤษ)


✍️ **คำตอบ:**



## ขั้นตอนที่ 3 — การสกัดผลการตรวจด้วยรูปแบบการค้นหาสำเร็จรูป

**Regular expression** (*regex* — นิพจน์ปรกติ) คือรูปแบบการค้นหาข้อความ ซึ่งคล้ายกับคำสั่ง "Find" ในโปรแกรม Word แต่สามารถระบุ *ประเภท* ของข้อความได้ (เช่น "ตัวเลขใด ๆ") นอกเหนือจากคำที่ตรงกันทุกตัวอักษร ในปฏิบัติการนี้ผู้เรียน **ไม่จำเป็นต้อง** เขียน regex เอง ความหมายของรูปแบบที่ใช้มีดังนี้

| ส่วนของรูปแบบ | ความหมาย |
|---|---|
| `^` | ที่ตำแหน่งต้นบรรทัด … |
| `(Hemoglobin\|Creatinine\|…)` | … มีชื่อการตรวจชื่อใดชื่อหนึ่งในรายการที่กำหนด (`\|` หมายถึง "หรือ") … |
| `\s+` | … ตามด้วยช่องว่างอย่างน้อย 1 ช่อง … |
| `(\d+(?:\.\d+)?)` | … ตามด้วยตัวเลข เช่น `142` หรือ `1.8` (**ผลการตรวจ**) … |
| `\s+(\S+)` | … ตามด้วยช่องว่างและกลุ่มอักขระที่ไม่มีช่องว่าง เช่น `mg/dL` (**หน่วย**) |

ส่วนที่อยู่ในวงเล็บ `( )` คือส่วนที่ระบบเก็บไว้ ตัวอย่างเช่น บรรทัด `Creatinine 1.8 mg/dL 0.7-1.3` จะได้ผลลัพธ์ `("Creatinine", "1.8", "mg/dL")`
เนื่องจากรูปแบบนี้ค้นหาเฉพาะชื่อที่อยู่ใน **รายการที่กำหนด** การตรวจที่ไม่อยู่ในรายการจะถูกละเว้นโดยไม่มีการแจ้งเตือน ซึ่งเป็นข้อผิดพลาดที่พบบ่อยในการใช้งานจริง


In [ ]:
# ⭐ Exercise 2 — แบบฝึกหัดที่ 2: การตรวจสอบว่าสกัดผลการตรวจได้ครบทุกรายการ
# ให้รันเซลล์นี้ แล้วเปรียบเทียบตารางผลลัพธ์กับภาพรายงานในขั้นตอนที่ 1
# ผลการตรวจขาดไป 1 รายการ เนื่องจากชื่อการตรวจนั้นไม่อยู่ในรายการด้านล่าง
# ➜ ให้เพิ่มชื่อการตรวจที่ขาดไปใน KNOWN_TESTS (สะกดให้ตรงกับรายงาน อยู่ภายในเครื่องหมายคำพูด "..."
#   และคั่นด้วยเครื่องหมายจุลภาค ,) จากนั้นรันเซลล์อีกครั้ง ผลลัพธ์ที่ถูกต้องมี 4 แถว
KNOWN_TESTS = ["Hemoglobin", "Glucose (fasting)", "Creatinine"]   # TODO เพิ่มชื่อการตรวจที่ขาดไป

def extract_labs(text):
    """ค้นหา 'ชื่อการตรวจ  ผล  หน่วย' ในแต่ละบรรทัดของข้อความ สำหรับทุกชื่อใน KNOWN_TESTS"""
    names = "|".join(re.escape(n) for n in KNOWN_TESTS)            # เชื่อมชื่อการตรวจด้วย | (หมายถึง "หรือ")
    pattern = re.compile(rf"^\s*({names})\s+(\d+(?:\.\d+)?)\s+(\S+)", re.M | re.I)
    spelling = {n.lower(): n for n in KNOWN_TESTS}                   # ใช้ตัวสะกดของชื่อตามรายการที่กำหนด
    rows = [(spelling[t.lower()], float(v), u) for t, v, u in pattern.findall(text)]
    return pd.DataFrame(rows, columns=["test", "result", "unit"])

labs = extract_labs(ocr_text)
labs


In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
KNOWN_TESTS = ["Hemoglobin", "Glucose (fasting)", "Creatinine", "LDL cholesterol"]

def extract_labs(text):
    """ค้นหา 'ชื่อการตรวจ  ผล  หน่วย' ในแต่ละบรรทัดของข้อความ สำหรับทุกชื่อใน KNOWN_TESTS"""
    names = "|".join(re.escape(n) for n in KNOWN_TESTS)            # เชื่อมชื่อการตรวจด้วย | (หมายถึง "หรือ")
    pattern = re.compile(rf"^\s*({names})\s+(\d+(?:\.\d+)?)\s+(\S+)", re.M | re.I)
    spelling = {n.lower(): n for n in KNOWN_TESTS}                   # ใช้ตัวสะกดของชื่อตามรายการที่กำหนด
    rows = [(spelling[t.lower()], float(v), u) for t, v, u in pattern.findall(text)]
    return pd.DataFrame(rows, columns=["test", "result", "unit"])

labs = extract_labs(ocr_text)
labs


💡 **คำอธิบายผลลัพธ์**

ข้อความที่ไม่มีโครงสร้างถูกแปลงเป็นตาราง โดยแต่ละแถวแทนการตรวจหนึ่งรายการ ข้อมูลในรูปแบบนี้สามารถนำไปเรียงลำดับ นำเสนอเป็นกราฟ หรือส่งเข้าสู่เวชระเบียนอิเล็กทรอนิกส์ (EHR) ได้ ตัวเลือก `re.I` กำหนดให้การค้นหาไม่คำนึงถึงตัวพิมพ์เล็กและตัวพิมพ์ใหญ่ ดังนั้น `HEMOGLOBIN` จะถูกค้นพบด้วย


## ขั้นตอนที่ 4 — การระบุค่าผิดปกติ

ระบบเปรียบเทียบผลการตรวจแต่ละรายการกับช่วงค่าอ้างอิง (reference range) ในคอลัมน์ REFERENCE ของรายงาน โดย `LOW` หมายถึงต่ำกว่าช่วงค่าอ้างอิง และ `HIGH` หมายถึงสูงกว่าช่วงค่าอ้างอิง


In [ ]:
# ⭐ Exercise 3 — แบบฝึกหัดที่ 3: การเพิ่มช่วงค่าอ้างอิงที่ขาดไป
# แต่ละรายการมีรูปแบบ  "ชื่อการตรวจ": (ค่าปกติต่ำสุด, ค่าปกติสูงสุด)
# รายการยังไม่มี Creatinine ผลลัพธ์จึงแสดงว่า "no range given" (ไม่มีช่วงค่าอ้างอิง)
# ➜ ให้อ่านช่วงค่าอ้างอิงของ Creatinine จากภาพรายงาน (คอลัมน์ REFERENCE) และเพิ่มไว้ด้านล่าง
#   ในรูปแบบเดียวกับบรรทัดอื่น จากนั้นรันเซลล์ ผลลัพธ์ที่ถูกต้องคือ Creatinine 1.8 ถูกระบุเป็น HIGH
# (สำหรับ LDL รายงานระบุว่า "<100" หมายความว่าค่า 0 ถึง 99 อยู่ในเกณฑ์ปกติ)
REF_RANGES = {
    "Hemoglobin":        (13.5, 17.5),   # g/dL (เพศชายวัยผู้ใหญ่)
    "Glucose (fasting)": (70, 99),       # mg/dL
    "LDL cholesterol":   (0, 99),        # mg/dL
    # TODO เพิ่ม Creatinine ในบรรทัดนี้
}

def flag(row):
    if row["test"] not in REF_RANGES:
        return "no range given"
    low, high = REF_RANGES[row["test"]]
    if row["result"] < low:
        return "LOW"
    if row["result"] > high:
        return "HIGH"
    return "normal"

labs["flag"] = labs.apply(flag, axis=1)
labs


In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
REF_RANGES = {
    "Hemoglobin":        (13.5, 17.5),   # g/dL (เพศชายวัยผู้ใหญ่)
    "Glucose (fasting)": (70, 99),       # mg/dL
    "LDL cholesterol":   (0, 99),        # mg/dL
    "Creatinine":        (0.7, 1.3),     # mg/dL
}

def flag(row):
    if row["test"] not in REF_RANGES:
        return "no range given"
    low, high = REF_RANGES[row["test"]]
    if row["result"] < low:
        return "LOW"
    if row["result"] > high:
        return "HIGH"
    return "normal"

labs["flag"] = labs.apply(flag, axis=1)
labs


💡 **คำอธิบายผลลัพธ์**

กฎอย่างง่ายแปลงค่าตัวเลขเป็นการระบุค่าผิดปกติทางคลินิก ซึ่งเป็นหลักการเดียวกับการแสดงผลการตรวจเป็นสีแดงในระบบสารสนเทศห้องปฏิบัติการ (LIS) หรือ EHR อย่างไรก็ตาม ช่วงค่าอ้างอิงขึ้นกับอายุ เพศ และวิธีการตรวจของแต่ละห้องปฏิบัติการ ระบบที่ใช้งานจริงจึงควรอ่านช่วงค่าอ้างอิงที่ห้องปฏิบัติการ *นั้น* ระบุไว้ แทนการกำหนดค่าคงที่ในโปรแกรม

## ขั้นตอนที่ 5 — การสกัดรายการยา

ความหมายของรูปแบบการค้นหายาสำเร็จรูปมีดังนี้

| ส่วนของรูปแบบ | ความหมาย |
|---|---|
| `([A-Z][a-z]+)` | คำที่ขึ้นต้นด้วยอักษรตัวพิมพ์ใหญ่ เช่น `Metformin` (**ชื่อยา**) … |
| `\s+(\d+(?:\.\d+)?)` | … ตามด้วยช่องว่างและตัวเลข เช่น `500` หรือ `12.5` (**ขนาดยา**) … |
| `\s*mg(?![A-Za-z/])` | … ตามด้วยตัวอักษร `mg` (อาจมีช่องว่างคั่น) แต่ **ไม่รวม** `mg/dL` ซึ่งเป็นหน่วยของผลการตรวจ มิใช่ขนาดยา |


In [ ]:
# รูปแบบการค้นหายา: ชื่อยา + ขนาดยาในหน่วย mg
MED_RE = re.compile(r"([A-Z][a-z]+)\s+(\d+(?:\.\d+)?)\s*mg(?![A-Za-z/])")

def extract_meds(text):
    return pd.DataFrame([(n, float(d)) for n, d in MED_RE.findall(text)], columns=["name", "dose_mg"])

meds = extract_meds(ocr_text)
meds


In [ ]:
# ⭐ Exercise 4 — แบบฝึกหัดที่ 4: การทดสอบรูปแบบการค้นหายากับข้อความที่ผู้เรียนกำหนด
# ➜ ให้แทนข้อความภายในเครื่องหมายคำพูดด้วยบรรทัดรายการยาในลักษณะที่พบในหน่วยงานของผู้เรียน
#   เช่น "Medications: Amlodipine 5 mg OD; aspirin 81mg OD; Insulin glargine 20 units QHS"
#   ทดลองใช้ชื่อยาตัวพิมพ์เล็ก ขนาดยาในหน่วย units / g / mcg ชื่อการค้า ชื่อย่อ (HCTZ) หรือภาษาไทย
#   จากนั้นตรวจสอบว่ารูปแบบนี้ค้นพบยารายการใด และไม่พบยารายการใด
my_line = "Medications: ..."   # TODO กำหนดข้อความรายการยา
extract_meds(my_line)


In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
my_line = "Medications: Amlodipine 5 mg OD; aspirin 81mg OD; Insulin glargine 20 units QHS; Metformin 1 g BID"
extract_meds(my_line)   # พบเพียง Amlodipine: 'aspirin' เป็นตัวพิมพ์เล็ก insulin ใช้หน่วย units และ metformin ใช้หน่วย g


✍️ **คำถามที่ 2** จากผลลัพธ์ข้างต้น รูปแบบการค้นหาไม่พบยารายการใดในข้อความของผู้เรียน เพราะเหตุใด (โปรดอ้างอิงตารางคำอธิบายรูปแบบ) และหากนำรายการยาที่สกัดได้ในลักษณะนี้ไปใช้ในการทบทวนความสอดคล้องของรายการยา (medication reconciliation) อาจเกิดปัญหาทางคลินิกใด


✍️ **คำตอบ:**



## ขั้นตอนที่ 6 — การประเมินความถูกต้องของการสกัดสารสนเทศ และผลกระทบของข้อผิดพลาดจาก OCR

การประเมินความถูกต้องทำโดยเปรียบเทียบคู่ข้อมูล **(ชื่อการตรวจ, ผลการตรวจ)** ที่สกัดได้กับเฉลย ด้วยตัวชี้วัดต่อไปนี้

- **Precision** (ค่าพยากรณ์บวก) — สัดส่วนของค่าที่ระบบสกัดได้ซึ่งถูกต้อง (precision ต่ำ หมายถึงมีค่าที่ผิดถูกบันทึกลงในเวชระเบียน)
- **Recall** (ความไว หรือ sensitivity) — สัดส่วนของค่าที่มีอยู่จริงในรายงานซึ่งระบบค้นพบ (recall ต่ำ หมายถึงผลการตรวจตกหล่น)
- **F1** — ค่าเฉลี่ยฮาร์มอนิกของ precision และ recall (0 = ใช้งานไม่ได้, 1 = ถูกต้องสมบูรณ์) ตารางอันดับ 🏆 ใช้หลักการเดียวกันนี้

คู่ข้อมูลจะนับว่าถูกต้องเมื่อทั้งชื่อ **และ** ตัวเลขถูกต้องทุกหลักเท่านั้น เช่น ค่า `1.8` ที่อ่านเป็น `18` ถือว่าผิด ข้อความด้านล่างคือรายงานฉบับเดิมในรูปแบบ *ที่มีสัญญาณรบกวน* ซึ่งจำลองผล OCR ของเอกสารแฟกซ์คุณภาพต่ำ


In [ ]:
# เฉลยของรายงานในขั้นตอนที่ 1
ANSWER_LABS = [("Hemoglobin", 11.2), ("Glucose (fasting)", 142), ("Creatinine", 1.8), ("LDL cholesterol", 165)]

def pair_f1(true_pairs, found_pairs):
    """คำนวณ precision, recall และ F1 ของรายการคู่ (ชื่อ, ค่า) โดยไม่คำนึงถึงตัวพิมพ์เล็กและใหญ่ของชื่อ"""
    true = [(n.lower(), round(float(v), 3)) for n, v in true_pairs]
    found = [(n.lower(), round(float(v), 3)) for n, v in found_pairs]
    tp = 0
    for p in found:
        if p in true:
            true.remove(p); tp += 1
    precision = tp / len(found) if found else 0.0
    recall = tp / (tp + len(true)) if (tp + len(true)) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return round(precision, 2), round(recall, 2), round(f1, 2)

def lab_pairs(df):
    return list(zip(df["test"], df["result"]))

# ตัวอย่างผล OCR ของรายงานฉบับเดิมที่สแกนด้วยคุณภาพต่ำ
noisy_text = """CENTRAL HOSPITAL - LABORATORY REPORT
Patient: John Doe MRN: 001234 Age: 57 Sex: M
Date: 2026-10-17
TEST RESULT UNIT REFERENCE
Hemoglobin 11.2 g/dL 13.5-17.5
Glucose (fasting) l42 mg/dL 70-99
Creatinine 1.B mg/dL 0.7-1.3
LDL cholesterol I65 mg/dL <100
Medications: Metformin 5O0 mg BID; Atorvastatin 20 mg OD
Impression: Anemia, hyperglycemia, elevated LDL."""

def fix_ocr_numbers(text, fixes):
    """แทนตัวอักษรที่มีรูปร่างคล้ายตัวเลขด้วยตัวเลข เฉพาะในคำที่มีลักษณะเป็นตัวเลข (เช่น 'l42', '5O0')"""
    chars = re.escape("".join(fixes))
    token = re.compile(rf"(?<![A-Za-z0-9])([0-9.{chars}]*[0-9][0-9.{chars}]*)(?![A-Za-z0-9])")
    return token.sub(lambda m: "".join(fixes.get(c, c) for c in m.group(1)), text)

print("OCR ภาพคมชัด         -> precision, recall, F1 =", pair_f1(ANSWER_LABS, lab_pairs(extract_labs(ocr_text))))
print("OCR มีสัญญาณรบกวน    -> precision, recall, F1 =", pair_f1(ANSWER_LABS, lab_pairs(extract_labs(noisy_text))))
extract_labs(noisy_text)


💡 **คำอธิบายผลลัพธ์**

ผลการตรวจเหลือเพียงแถวเดียว เนื่องจาก `l42` (ตัวอักษรแอล) `I65` (ตัวอักษรไอ) และ `1.B` (ตัวอักษรบี) ไม่ใช่ตัวเลข รูปแบบการค้นหาจึงข้ามบรรทัดเหล่านี้ ฟังก์ชัน `fix_ocr_numbers` แก้ไขตัวอักษรที่มีรูปร่างคล้ายตัวเลข **เฉพาะในคำที่มีลักษณะเป็นตัวเลข** ดังนั้นคำอย่าง "Hemoglobin" ซึ่งมีตัวอักษร `l` จะไม่ถูกแก้ไข


In [ ]:
# ⭐ Exercise 5 — แบบฝึกหัดที่ 5: การระบุและแก้ไขข้อผิดพลาดจาก OCR
# OCR_FIXES กำหนดว่าตัวอักษรที่มีรูปร่างคล้ายตัวเลขแต่ละตัวควรแทนด้วยตัวเลขใด
# ➜ ให้รันเซลล์และพิจารณาข้อความที่แสดง ยังมีผลการตรวจ 1 ค่าที่อ่านผิดและยังไม่ได้รับการแก้ไข (F1 < 1)
#   ให้ระบุค่าดังกล่าว แล้วเพิ่มรายการ  "ตัวอักษร": "ตัวเลข"  อีก 1 รายการใน OCR_FIXES จนค่า F1 เท่ากับ 1.0
#   (ค่า F1 = 1.0 ต้องมีชื่อการตรวจครบทั้ง 4 รายการจากแบบฝึกหัดที่ 2)
# คำแนะนำ: พิจารณาบรรทัด Creatinine ว่าตัวอักษรที่ปรากฏมีรูปร่างคล้ายตัวเลขใด
OCR_FIXES = {"O": "0", "o": "0", "l": "1", "I": "1"}   # TODO เพิ่มอีก 1 รายการ

fixed_text = fix_ocr_numbers(noisy_text, OCR_FIXES)
print(fixed_text, "\n")
print("OCR มีสัญญาณรบกวน      -> precision, recall, F1 =", pair_f1(ANSWER_LABS, lab_pairs(extract_labs(noisy_text))))
print("หลังแก้ไขด้วย OCR_FIXES -> precision, recall, F1 =", pair_f1(ANSWER_LABS, lab_pairs(extract_labs(fixed_text))))
extract_labs(fixed_text)


In [ ]:
# @title ✅ เฉลย — หากไม่สามารถทำแบบฝึกหัดได้ ให้รันเซลล์นี้ (ดับเบิลคลิกเพื่อแสดงโค้ด) { display-mode: "form" }
OCR_FIXES = {"O": "0", "o": "0", "l": "1", "I": "1", "B": "8"}

fixed_text = fix_ocr_numbers(noisy_text, OCR_FIXES)
print(fixed_text, "\n")
print("OCR มีสัญญาณรบกวน      -> precision, recall, F1 =", pair_f1(ANSWER_LABS, lab_pairs(extract_labs(noisy_text))))
print("หลังแก้ไขด้วย OCR_FIXES -> precision, recall, F1 =", pair_f1(ANSWER_LABS, lab_pairs(extract_labs(fixed_text))))
extract_labs(fixed_text)


💡 **คำอธิบายผลลัพธ์**

อักขระที่อ่านผิดเพียงตัวเดียวทำให้ค่า creatinine **1.8 mg/dL** (ซึ่งผิดปกติ) หายไปจากตาราง กฎการทำความสะอาดข้อมูลอย่างง่ายช่วยกู้ค่าดังกล่าวกลับมา และทำให้ค่า F1 เพิ่มเป็น 1.0 อย่างไรก็ตาม กฎการทำความสะอาดข้อมูลอาจ *ก่อให้เกิด* ข้อผิดพลาดได้เช่นกัน (ตัวอักษร `B` ที่ถูกต้องอาจถูกเปลี่ยนเป็น `8`) ค่าที่สกัดได้จึงควรผ่านการตรวจสอบความสมเหตุสมผล (plausibility check) เช่น ค่า creatinine อยู่ในช่วงที่เป็นไปได้ทางสรีรวิทยาหรือไม่ และควรได้รับการทบทวนโดยบุคลากรก่อนนำไปใช้ทางคลินิก


✍️ **คำถามที่ 3** หาก OCR อ่านจุดทศนิยมตกหล่น ทำให้ค่า hemoglobin `11.2` กลายเป็น `112` g/dL หรือค่า creatinine `1.8` สูญหายไป จงอธิบายผลกระทบที่อาจเกิดขึ้นต่อผู้ป่วย และเสนอ **มาตรการป้องกัน 2 ประการ** ที่โรงพยาบาลควรกำหนดเมื่อใช้ระบบสกัดสารสนเทศด้วย OCR แบบอัตโนมัติ


✍️ **คำตอบ:**



## ส่วนเสริม (ไม่บังคับ) — การสกัดสารสนเทศด้วยโมเดล NLP ที่เรียนรู้จากข้อมูล (NER)

กฎที่เขียนขึ้นเองมีข้อจำกัด เนื่องจากต้องเพิ่มกฎใหม่ทุกครั้งที่พบตัวสะกด หน่วย หรือรูปแบบเอกสารที่แตกต่างไป ระบบในปัจจุบันจึงใช้โมเดล **NER** (*Named-Entity Recognition* — การรู้จำชื่อเฉพาะ) ร่วมด้วย โมเดลประเภทนี้เรียนรู้จากตัวอย่างที่มีการกำกับป้าย (label) จำนวนมาก เพื่อระบุว่าคำใดเป็น *ยา* *ขนาดยา* *ผลการตรวจ* หรือ *โรค* แม้ในประโยคที่ไม่เคยพบมาก่อน ผู้เรียนที่สนใจสามารถทดลองใช้ [scispaCy](https://allenai.github.io/scispacy/) หรือโมเดล NER ทางชีวการแพทย์บน Hugging Face (ขนาดดาวน์โหลดประมาณ 250 MB; ใช้ CPU ได้) โดยคัดลอกโค้ดด้านล่างไปวางในเซลล์โค้ดใหม่ (**+ Code**)

```python
!pip -q install transformers
from transformers import pipeline
ner = pipeline("token-classification", model="d4data/biomedical-ner-all", aggregation_strategy="simple")
ner("Metformin 500 mg BID for type 2 diabetes; hemoglobin 11.2 g/dL")
```

ผลลัพธ์แต่ละรายการประกอบด้วย `entity_group` (เช่น `Medication`, `Dosage`, `Lab_value`) และคำที่ตรวจพบใน `word` ทั้งนี้ โมเดล NER อาจผิดพลาดได้ และยังต้องอาศัยข้อความจาก OCR ที่มีคุณภาพเพียงพอ


✍️ **คำถามที่ 4 (คำถามสะท้อนคิดหลัก)** จงอธิบายโดยสังเขป (2–4 ประโยค) ว่าในปฏิบัติการนี้ OCR และกฎที่เขียนขึ้นเองมีข้อจำกัดในประเด็นใด (เช่น หน่วย รูปแบบเอกสาร อักขระที่มีรูปร่างคล้ายกัน ตัวสะกด หรือลายมือ) และโมเดล NER ที่เรียนรู้จากข้อมูล หรือขั้นตอนการทบทวนโดยบุคลากร (human-in-the-loop) ช่วยลดข้อจำกัดดังกล่าวได้อย่างไร


✍️ **คำตอบ:**



## 🏆 โจทย์ท้าทายบนตารางอันดับ (Leaderboard) — ไม่บังคับ สำหรับผู้เรียนที่เขียนโปรแกรมได้

> **ส่วนนี้ไม่บังคับ และไม่นับเป็นส่วนหนึ่งของชิ้นงานที่ต้องส่ง** ผู้เรียนที่ไม่เข้าร่วมสามารถข้ามไปยังหัวข้อ *ชิ้นงานที่ต้องส่ง*

ให้ทำ OCR กับ **ภาพรายงานผลการตรวจทางห้องปฏิบัติการสังเคราะห์ 20 ฉบับ** ของรายวิชา และสกัด (1) การตรวจทุกรายการพร้อมผลการตรวจที่เป็นตัวเลข และ (2) ยาทุกรายการพร้อมขนาดยาในหน่วย mg **ตัวชี้วัด: ค่าเฉลี่ยของ F1 ของคู่ (ชื่อการตรวจ, ผลการตรวจ) และ F1 ของคู่ (ชื่อยา, ขนาดยา)** (ชื่อการตรวจเปรียบเทียบหลังแปลงเป็นตัวพิมพ์เล็กและตัดส่วนตั้งแต่เครื่องหมาย `(` ออก ส่วนชื่อยาเปรียบเทียบด้วยชื่อสามัญทางยา)

รายงานชุดทดสอบมี **รายการตรวจและหน่วยที่หลากหลายกว่า** ตัวอย่างข้างต้น (`%`, `mmol/L`, `U/L`, `10^3/uL` เป็นต้น) และบางฉบับมีภาพเอียงเล็กน้อยหรือมีสัญญาณรบกวน โค้ดตั้งต้น (baseline) ด้านล่างสามารถรันได้ทันที และสร้างไฟล์ `lab07_submission.json` ในรูปแบบที่ถูกต้อง

`{"R01": {"labs": [{"test": "Glucose (fasting)", "result": 95, "unit": "mg/dL"}, ...], "medications": [{"name": "Metformin", "dose_mg": 850}, ...]}, ...}`

การส่งผลลัพธ์ ให้ระบุ `NAME` และ `JOIN_CODE` ในเซลล์ ⚙️ การตั้งค่า (ผู้สอนประกาศรหัสเข้าร่วมในการเรียนการสอนผ่าน Zoom) แล้วรันเซลล์ดังกล่าวอีกครั้ง

**หลักความซื่อสัตย์ทางวิชาการ (honor code):** ให้ส่งเฉพาะผลการทำนายที่ได้จากโปรแกรมของผู้เรียนเองซึ่งประมวลผลจากภาพ ห้ามพิมพ์หรือแก้ไขผลของรายงานชุดทดสอบด้วยตนเอง


In [ ]:
# 🏆 Baseline — ทำ OCR และสกัดสารสนเทศจากรายงานทั้ง 20 ฉบับ สร้างไฟล์ lab07_submission.json และส่งผลลัพธ์
import json

manifest = pd.read_csv(lab_data("ocr/manifest.csv"))

# รูปแบบการค้นหาผลการตรวจแบบทั่วไป (ไม่จำกัดเฉพาะชื่อในรายการที่กำหนด): บรรทัดที่ขึ้นต้นด้วยชื่อการตรวจ
# (ตัวอักษร/ตัวเลข/ช่องว่าง/()/-) ตามด้วยตัวเลข และตามด้วยหน่วย
LAB_RE = re.compile(r"^([A-Za-z][A-Za-z0-9 ()/-]*?)\s+(\d+(?:\.\d+)?)\s+(\S+)", re.M)
MED_RE = re.compile(r"([A-Z][a-z]+)\s+(\d+(?:\.\d+)?)\s*mg(?![A-Za-z/])")

def ocr_image(path):
    return pytesseract.image_to_string(Image.open(path), config="--psm 6")

def extract_report(text):
    text = fix_ocr_numbers(text, OCR_FIXES)
    labs = [{"test": t.strip(), "result": float(v), "unit": u} for t, v, u in LAB_RE.findall(text)]
    meds = [{"name": n, "dose_mg": float(d)} for n, d in MED_RE.findall(text)]
    return {"labs": labs, "medications": meds}

submission = {}
for rid, f in zip(manifest["id"], manifest["file"]):
    submission[rid] = extract_report(ocr_image(lab_data(f"ocr/{f}")))

print(json.dumps(submission["R01"], indent=1))
with open("lab07_submission.json", "w") as fh:
    json.dump(submission, fh, indent=1)
print(f"สร้างไฟล์ lab07_submission.json เรียบร้อย ({len(submission)} รายงาน)")

if NAME and JOIN_CODE:
    submit_to_leaderboard("lab07_submission.json")
else:
    print("หากต้องการส่งผลไปยังตารางอันดับ โปรดระบุ NAME และ JOIN_CODE ในเซลล์การตั้งค่า แล้วรันเซลล์ดังกล่าวอีกครั้ง")


### แนวทางการพัฒนาให้ได้คะแนนสูงกว่า baseline

- **วิเคราะห์ข้อผิดพลาดก่อน:** แสดงข้อความ OCR ของรายงานบางฉบับ (`print(ocr_image(lab_data("ocr/reports/R05.png")))`) และเปรียบเทียบกับภาพต้นฉบับ
- **การเตรียมภาพก่อนทำ OCR (preprocessing):** แปลงเป็นภาพระดับเทา (grayscale) ขยายภาพ 2 เท่า (`img.resize(...)`) แปลงเป็นภาพสองระดับ (thresholding) และปรับภาพสแกนที่เอียงให้ตรง (deskew)
- **การปรับค่าพารามิเตอร์ของ Tesseract:** ทดลองโหมด `--psm` อื่น หรือ `-c tessedit_char_whitelist=...` สำหรับคอลัมน์ผลการตรวจ
- **การเพิ่มกฎแก้ไขข้อผิดพลาดจาก OCR** ใน `OCR_FIXES` (`S`↔`5`, `Z`↔`2`, `g`↔`9`) โดยตรวจสอบว่าไม่ทำให้ค่าที่ถูกต้องอยู่แล้วเกิดความผิดพลาด
- **การปรับปรุงรูปแบบการค้นหาผลการตรวจ:** ใช้รายการชื่อการตรวจที่กำหนดไว้ (ดังขั้นตอนที่ 3) หรือการจับคู่ชื่อที่สะกดผิดแบบใกล้เคียง (fuzzy matching, `difflib.get_close_matches`) ร่วมกับการตรวจสอบความสมเหตุสมผลของค่า
- **การสกัดรายการยา:** จับคู่กับพจนานุกรมยาโดยไม่คำนึงถึงตัวพิมพ์เล็กและใหญ่ หรือใช้โมเดล NER (ดูส่วนเสริม NER)


## ชิ้นงานที่ต้องส่ง (งานเดี่ยว)

ให้ส่ง **โน้ตบุ๊กนี้ซึ่งรันครบทุกเซลล์ตามลำดับจากบนลงล่าง** (Runtime ▸ Run all) ตามช่องทางที่รายวิชากำหนด โดยโน้ตบุ๊กต้องแสดง

- ตารางผลการตรวจที่สกัดได้พร้อมการระบุค่าผิดปกติ และรายการยาที่สกัดได้
- คำตอบ ✍️ ของคำถามที่ 1–4

ผู้เรียนที่เข้าร่วมโจทย์ท้าทายบนตารางอันดับ 🏆 สามารถระบุชื่อที่ใช้บนตารางอันดับและคะแนนสูงสุดไว้ในส่วนนี้ (ไม่บังคับ)

✍️ **ชื่อบนตารางอันดับ / คะแนน (ไม่บังคับ):**
